# Experiment 6: Comprehensive Caching Benchmark

## Objective

Evaluate the performance of different caching strategies using the complete SEVA_AI golden test dataset containing 973 questions.

We will compare uncached and cached RAG execution to determine the impact on latency, resource utilization, API costs and answer quality.

## Benchmark configurations

1. No caching (baseline).
2. Embedding caching.
3. Retrieval caching.
4. Parent document caching.
5. LLM response caching.
6. Combined caching.

## Evaluation metrics

- Average latency.
- P50, P95 and P99 latency.
- Cache hit rate and miss rate.
- Redis memory consumption.
- Embedding generation time.
- Retrieval time.
- LLM generation time.
- Token consumption and estimated API cost.
- Answer correctness and faithfulness.

## Dataset

Golden test dataset: 973 questions.

All benchmark configurations must use the same questions and consistent retrieval and generation settings.

## 1. Load Golden Test Dataset

Load the existing 973-question golden test dataset.

Purpose:
- Use identical queries across benchmark configurations.
- Preserve ground-truth answers and relevant scheme IDs.
- Establish a consistent evaluation dataset.
- Avoid changing the original golden dataset.

In [2]:
import json
import pandas as pd
from pathlib import Path

GOLDEN_TEST_PATH = Path(
    "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\final\\Final_dataset_splits\\test.json"
)

with open(GOLDEN_TEST_PATH, "r", encoding="utf-8") as f:
    golden_data = json.load(f)

golden_df = pd.DataFrame(golden_data)

print("Total test questions:", len(golden_df))
print("\nColumns:", golden_df.columns.tolist())
print("\nSample:")
display(golden_df.head(3))

Total test questions: 973

Columns: ['test_id', 'faq_id', 'scheme_id', 'scheme_name', 'question', 'ground_truth_answer', 'categories', 'level', 'source_url', 'relevant_scheme_ids', 'relevant_doc_ids', 'answerable']

Sample:


,test_id,faq_id,scheme_id,scheme_name,question,ground_truth_answer,categories,level,source_url,relevant_scheme_ids,relevant_doc_ids,answerable
0,GD_000031,0Be1F5gB5uF80_MGnI9U_FAQ_001,0Be1F5gB5uF80_MGnI9U,Dayanand Bandodkar Scheme For Higher Education...,Is this scheme limited to certain fields of st...,"The scheme covers a wide range of fields, incl...",[Education & Learning],State,https://www.myscheme.gov.in/schemes/dbsheo,[0Be1F5gB5uF80_MGnI9U],[],True
1,GD_004164,dxfCF5gB5uF80_MGAJJb_FAQ_006,dxfCF5gB5uF80_MGAJJb,PMMSY: Construction of Fish Retail Markets inc...,Is there a limit to how many units of central ...,"Yes, central assistance is restricted to one u...","[Agriculture,Rural & Environment, Business & E...",State,https://www.myscheme.gov.in/schemes/pmmsycrmhr,[dxfCF5gB5uF80_MGAJJb],[],True
2,GD_001489,FBfZF5gB5uF80_MGUpOf_FAQ_020,FBfZF5gB5uF80_MGUpOf,Accidental Benefit,What form needs to be used to apply for this w...,The application must be made in the prescribed...,[Social welfare & Empowerment],State,https://www.myscheme.gov.in/schemes/adwb,[FBfZF5gB5uF80_MGUpOf],[],True


In [3]:
required_columns = [
    "test_id",
    "question",
    "ground_truth_answer",
    "relevant_scheme_ids",
    "answerable"
]

missing_columns = [
    col for col in required_columns
    if col not in golden_df.columns
]

print("Missing columns:", missing_columns)
print("Total records:", len(golden_df))
print("Unique test IDs:", golden_df["test_id"].nunique())
print("Missing questions:", golden_df["question"].isna().sum())
print("Answerable questions:", golden_df["answerable"].sum())

Missing columns: []
Total records: 973
Unique test IDs: 973
Missing questions: 0
Answerable questions: 973


## 3. Baseline Retrieval Without Caching

Execute all 973 golden test questions directly against the existing Chroma discovery index.

**Configuration:**
- Vector database: Chroma
- Collection: seva_discovery_v1
- Candidate chunks: Top 30
- Unique scheme candidates: Top 10
- Redis caching: Disabled
- Embedding cache: Disabled

**Purpose:**
- Measure baseline retrieval latency.
- Identify whether the relevant scheme appears in retrieved results.
- Establish Recall@5, Recall@10, MRR and NDCG@10.
- Create a reference dataset for future caching comparisons.

We will aggregate retrieved chunks by scheme_id and preserve their original ranking.

In [4]:
import json
import time
from pathlib import Path
from collections import defaultdict

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [5]:
OUTPUT_DIR1 = Path("G:\\SEVA_Ai\\SEVA_Ai\\output\\SEVA_CHILDREN")

VECTORSTORE_DIR = OUTPUT_DIR1 / "vectorstores"

PARENT_PATH = OUTPUT_DIR1 / "parent_documents.jsonl"

print("Parent file exists:", PARENT_PATH.exists())
print("Vectorstore directory exists:", VECTORSTORE_DIR.exists())

Parent file exists: True
Vectorstore directory exists: True


In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 16
    }
)

print("Embedding model loaded successfully.")

Could not reach the Hub ([Errno 11001] getaddrinfo failed). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/2_Normalize/config.json
Retrying in 1s [Retry 1/5].


Embedding model loaded successfully.


In [7]:
discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "discovery")
)

detail_db = Chroma(
    collection_name="seva_detail_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "detail")
)

discovery_count = discovery_db._collection.count()
detail_count = detail_db._collection.count()

print("Discovery collection:", discovery_count)
print("Detail collection:", detail_count)
print("Total indexed chunks:", discovery_count + detail_count)

Discovery collection: 23438
Detail collection: 67945
Total indexed chunks: 91383


In [8]:
import time
import numpy as np
from collections import defaultdict

def baseline_retrieval(query, chunk_k=30, scheme_k=10):

    start = time.perf_counter()

    results = discovery_db.similarity_search_with_score(
        query,
        k=chunk_k
    )

    # Aggregate chunks by scheme ID
    scheme_scores = {}

    for doc, distance in results:
        scheme_id = doc.metadata.get("scheme_id")

        if scheme_id is None:
            continue

        if (
            scheme_id not in scheme_scores
            or distance < scheme_scores[scheme_id]
        ):
            scheme_scores[scheme_id] = distance

    # Rank unique schemes by minimum distance
    ranked_schemes = sorted(
        scheme_scores.items(),
        key=lambda x: x[1]
    )

    ranked_scheme_ids = [
        scheme_id
        for scheme_id, score in ranked_schemes[:scheme_k]
    ]

    latency = (time.perf_counter() - start) * 1000

    return {
        "ranked_scheme_ids": ranked_scheme_ids,
        "retrieved_chunks": len(results),
        "unique_schemes": len(scheme_scores),
        "latency_ms": latency
    }

## 4. Execute Baseline Retrieval on Golden Dataset

Run the uncached retrieval function for all 973 test questions.

**Important:**
- Use the same queries for every future benchmark configuration.
- Do not use Redis embedding or retrieval caches.
- Record individual query latency.
- Preserve retrieved scheme rankings for evaluation.

The output will be our baseline retrieval results.

In [9]:
baseline_results = []

for idx, row in golden_df.iterrows():

    result = baseline_retrieval(
        query=row["question"],
        chunk_k=30,
        scheme_k=10
    )

    baseline_results.append({
        "test_id": row["test_id"],
        "question": row["question"],
        "ground_truth_scheme_ids": row["relevant_scheme_ids"],
        **result
    })

    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/973 questions")

baseline_df = pd.DataFrame(baseline_results)

print("\nTotal processed:", len(baseline_df))
print("Average latency:", round(baseline_df["latency_ms"].mean(), 2), "ms")

Processed 100/973 questions
Processed 200/973 questions
Processed 300/973 questions
Processed 400/973 questions
Processed 500/973 questions
Processed 600/973 questions
Processed 700/973 questions
Processed 800/973 questions
Processed 900/973 questions

Total processed: 973
Average latency: 352.98 ms


## 5. Inspect Baseline Retrieval Results

Inspect the retrieved scheme IDs and latency for individual queries.

Verify that:
- All 973 questions were processed.
- Retrieved scheme IDs are populated.
- Retrieval latency is recorded.
- Ground-truth scheme IDs are preserved.

In [10]:
display(
    baseline_df[
        [
            "test_id",
            "retrieved_chunks",
            "unique_schemes",
            "ground_truth_scheme_ids",
            "ranked_scheme_ids",
            "latency_ms"
        ]
    ].head(10)
)

print("\nMissing retrieval results:",
      baseline_df["ranked_scheme_ids"].isna().sum())

print("Total retrieved chunks:",
      baseline_df["retrieved_chunks"].sum())

,test_id,retrieved_chunks,unique_schemes,ground_truth_scheme_ids,ranked_scheme_ids,latency_ms
0,GD_000031,30,27,[0Be1F5gB5uF80_MGnI9U],"[3ca-F5gB0AGc0F0iYTUA, cRhQHJ4B5uF80_MGdWqo, N...",1027.1135
1,GD_004164,30,30,[dxfCF5gB5uF80_MGAJJb],"[Zce6R5wB0AGc0F0ijAOI, AMa5F5gB0AGc0F0iwjVE, m...",461.5187
2,GD_001489,30,24,[FBfZF5gB5uF80_MGUpOf],"[Zca7F5gB0AGc0F0i0TVo, SBe7F5gB5uF80_MG1JE_, o...",501.5371
3,GD_002828,30,21,[RaE9GJgBXu-iUKXqCHkv],"[0xe-F5gB5uF80_MGZZFD, 36G_F5gBXu-iUKXqgXY5, g...",406.9232
4,GD_000477,30,30,[5MbVF5gB0AGc0F0isDb-],"[8KJMo58BXu-iUKXqflrJ, pcZFp5oB0AGc0F0iIaoe, C...",346.1603
5,GD_003934,30,26,[bKFTJJoBXu-iUKXqjHrW],"[SqE9GJgBXu-iUKXqCHmc, nBhD4JsB5uF80_MGDVsg, T...",467.4879
6,GD_002406,30,26,[NRezF5gB5uF80_MGGo-5],"[Sce1cp4B0AGc0F0inRMp, 8ccTkZ4B0AGc0F0i9hMs, A...",513.7423
7,GD_000169,30,19,[1selQJ8B0AGc0F0izxWD],"[QcZklJkB0AGc0F0igzlj, 3Be1F5gB5uF80_MGnY_A, 7...",516.6341
8,GD_006084,30,25,[wKL_z5oBXu-iUKXqtkCg],"[T6G3F5gBXu-iUKXqM3Uy, taG9F5gBXu-iUKXq4na1, z...",712.5943
9,GD_000536,30,23,[66I40ZwBXu-iUKXqEEk4],"[66I40ZwBXu-iUKXqEEk4, E6G_F5gBXu-iUKXq8nfc, D...",843.4181



Missing retrieval results: 0
Total retrieved chunks: 29190


## 6. Evaluate Baseline Retrieval Quality

Calculate retrieval quality using the complete golden dataset.

### Metrics

1. Recall@5: Percentage of relevant schemes retrieved in the top 5.
2. Recall@10: Percentage of relevant schemes retrieved in the top 10.
3. MRR@10: Measures the reciprocal rank of the first relevant scheme.
4. NDCG@10: Measures the ranking quality using graded or binary relevance.

For this experiment, use binary relevance based on `relevant_scheme_ids`.

These metrics establish the retrieval quality baseline before introducing caching.

## 6. Initialize Ragas and Langfuse Evaluation

Prepare the Ragas evaluation metrics and Langfuse experiment environment.

We will use Ragas for retrieval and answer quality evaluation, and Langfuse to store and compare experiment results.

The evaluation will use all 973 golden test questions.

In [12]:
import importlib.metadata as metadata

for package in ["ragas", "langfuse", "datasets"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: Not installed")

# Verify Ragas metric imports
from ragas.metrics import (
    IDBasedContextPrecision,
    IDBasedContextRecall
)

from ragas.dataset_schema import SingleTurnSample

print("\nRagas evaluation imports successful!")

ragas: 0.4.4
langfuse: 5.2.0
datasets: 5.2.0

Ragas evaluation imports successful!


C:\Users\devendra\AppData\Local\Temp\ipykernel_9968\2701990353.py:10: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (
C:\Users\devendra\AppData\Local\Temp\ipykernel_9968\2701990353.py:10: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0 and has no direct equivalent in 'ragas.metrics.collections'.
  from ragas.metrics import (


## 6. Ragas-Based Retrieval Evaluation

Evaluate baseline retrieval quality using Ragas built-in ID-based metrics.

### Metrics

1. ID-Based Context Precision
2. ID-Based Context Recall

### Evaluation methodology

- Retrieved context IDs: Retrieved scheme IDs from baseline retrieval.
- Reference context IDs: Ground-truth relevant scheme IDs.
- Dataset: All 973 golden test questions.

These metrics evaluate scheme-level retrieval quality.

Note: This is scheme-level evaluation, not child-document-level evaluation. Our current golden dataset does not contain relevant document IDs.